In [0]:
!pip install typing
!pip install openai
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install "pyarrow<20" "numpy<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/78.6 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=0fc9b841bda3d90bf760cab5e8be5976bc45ecafbb4276aee3444de221352d5d
  Stored in directory: /home/spark-cae5c231-5dbc-456c-be18-1f/.cache/pip/wheels/9d/67/2f/53e3ef32ec48d11d7d60245255e2d71e908201d20c880c08ee
Successfully built typing
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Obtaining dependency information for langchain from https://files.pythonhosted.org/packages/ed/5c/5c0be747261e1f8129b875fa3bfea736bc5fe17652f9d5e15ca118571b6f/langchain-0.3.25-py3-

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import json
from datetime import datetime
from typing import Optional, List
import openai
from openai import AzureOpenAI
from pydantic import ValidationError
from dotenv import load_dotenv
from pathlib import Path
from openai import AzureOpenAI
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import AzureAIDocumentIntelligenceLoader

**PDF extraction using RAG In-Memory**

In [0]:
def load_env_vars(env_path: str):
    load_dotenv(dotenv_path=Path(env_path))
    return {
        "openai_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "openai_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),
        "openai_api_version": os.getenv("AZURE_OPENAI_API_VERSION"),
        "embedding_deployment": os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"),
        "chat_deployment": os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME"),
        "embedding_key": os.getenv("AZURE_OPENAI_EMBEDDING_API_KEY"),
        "embedding_endpoint": os.getenv("AZURE_OPENAI_EMBEDDING_ENDPOINT")          # Customize if needed
    }

def create_embeddings(config):
    return AzureOpenAIEmbeddings(
        azure_deployment=config["embedding_deployment"],
        azure_endpoint=config["embedding_endpoint"],
        api_key=config["embedding_key"],
    )

def load_and_split_document(file_path: str, config):         #document loader using azureaidocumentintelligence
    loader = AzureAIDocumentIntelligenceLoader(api_endpoint="https://openai-genai-sca-dev-eastus-01.openai.azure.com/", api_key="4def9069357c408a9cc50c88673f8e36", file_path=file_path, api_model="prebuilt-layout")
    documents = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=3000, chunk_overlap=0)
    return text_splitter.split_documents(documents)



def setup_vector_store(embeddings, docs):
    vector_store = InMemoryVectorStore(embeddings)          #In memory vector store embeddings
    vector_store.add_documents(docs)
    return vector_store

def create_llm(config):                                    #gpt-4.1 model
    return AzureChatOpenAI(
        openai_api_version=config["openai_api_version"],
        azure_deployment=config["chat_deployment"],
        azure_endpoint=config["openai_endpoint"],
        api_key=config["openai_key"],
        temperature=0,
    )

def create_rag_chain(llm, vector_store):
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

def get_query_result(rag_chain, query: str):
    return rag_chain.invoke({"query": query})


def main():
    print("Starting RAG process...")


    # .env path
    config = load_env_vars("/Workspace/Users/znidhishree.sanam@wescodist.com/IDP/License_Manager/secrets.env")
    embeddings = create_embeddings(config)
    print(config)
    #Path of pdf document in blob storage
    docs = load_and_split_document("/Volumes/dev_datascience/volumes/ds-idp-volume/LM_PDFs/email 4.pdf", config)       #change the file path

    vector_store = setup_vector_store(embeddings, docs)
    llm = create_llm(config)
    rag_chain = create_rag_chain(llm, vector_store)

    query = """ Extract appropriate fields and generate json as per the following schema and keep the json formatting as code:
    {
	"headers": {
		"vendorname": string
	},
	"orderdetails": {
		"documentDate": datetime, 
        // This refers to the invoice or document date. Use the field labeled "Invoice Date", "Date of Invoice", "Document Date", etc. 
        // Do not use "Ship Date", "Due Date", or unrelated dates. Format: YYYY-MM-DD.
		
        "vendorSoNumber": string,  
        // This is the Sales Order Number — not a shipment or delivery number. Look for fields labeled "Sales Order", "SO#", "Order#" etc. 
        // Ignore fields labeled "Shipment Number", "Shipping ID", etc.
		
        "vendorPoNumber": string,
        // This is the Purchase Order Number — usually labeled "Purchase Order", "PO#", etc. 
        // Do not extract shipment-related numbers.
		
        "endCustomerName": string,
	},
	"licensedetails": [
		{
			"partno": string,
            // This refers to the Part Number or SKU ID. Extract only values containing uppercase A–Z, numbers 0–9, or hyphens (-). 
            // Ignore non-English characters, Cyrillic text, or special characters.
			
            "quantity": int,
			
            "licenseKey": string,
            // This may not be explicitly labeled. If a string resembling a license key (usually alphanumeric, often with dashes or unique formatting) 
            // appears near the quantity or part number, extract it as licenseKey. If no such key is present nearby, return null.
			
            "startdate": datetime,
			"enddate": datetime
		}
	],
	"others": {
		"slcdetails": [
			{
				"slcId": string,
                // Just extract the slcid which is in alphanumeric format with hyphens.

				"carePlusExpDate": datetime,
				"carePremExpDate": datetime,
				"deviceQty": int
			}
		],
        // Optional. Only extract SLC details if they are present. 

		"SERIALS SHIPPED": [string]
        // Optional. If present, extract a list of serial numbers (strings). These may appear under headings like "Serials Shipped", "Serial Numbers" or similar
        // Extract the full string value associated with each serial, including any descriptive or parenthetical text
        // If not present, return an empty list.
	}
    }
    """

    result = get_query_result(rag_chain, query)

    #print("\nQuery:", query)
    print("\nAnswer:", result["result"])

if __name__ == "__main__":
    main()


Starting RAG process...
{'openai_key': '', 'openai_endpoint': 'https://openai-genai-sca-dev-eastus-01.openai.azure.com/', 'openai_api_version': '2024-12-01-preview', 'embedding_deployment': 'text-embedding-3-large', 'chat_deployment': 'gpt-4.1', 'embedding_key': '', 'embedding_endpoint': 'https://openai-genai-sca-dev-eastus-01.openai.azure.com/'}


---------------------------------------------------------------------------
ClientAuthenticationError                 Traceback (most recent call last)
File <command-2315930052461245>, line 133
    130     print("\nAnswer:", result["result"])
    132 if __name__ == "__main__":
--> 133     main()

File <command-2315930052461245>, line 64, in main()
     62 print(config)
     63 #Path of pdf document in blob storage
---> 64 docs = load_and_split_document("/Volumes/dev_datascience/volumes/ds-idp-volume/LM_PDFs/email 4.pdf", config)       #change the file path
     66 vector_store = setup_vector_store(embeddings, docs)
     67 llm = create_llm(config)

File <command-2315930052461245>, line 22, in load_and_split_document(file_path, config)
     20 def load_and_split_document(file_path: str, config):         #document loader using azureaidocumentintelligence
     21     loader = AzureAIDocumentIntelligenceLoader(api_endpoint="https://openai-genai-sca-dev-eastus-01.openai.azure.com/", api_key

**PDF extraction using Azure ai document intelligence loader**

In [0]:
from langchain_community.document_loaders import AzureAIDocumentIntelligenceLoader


load_dotenv(dotenv_path=Path('/Workspace/Users/znidhishree.sanam@wescodist.com/IDP/License_Manager/secrets_AI.env'))
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
key = os.getenv("AZURE_OPENAI_API_KEY")
version= os.getenv("AZURE_OPENAI_API_VERSION")
model_deployment = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")

#document loader for pdf file in blob storage
file_path = "/Volumes/dev_datascience/volumes/ds-idp-volume/LM_PDFs/email 4.pdf"

loader = AzureAIDocumentIntelligenceLoader(
    api_endpoint=endpoint, api_key=key, file_path=file_path, api_model="prebuilt-layout"
)
documents = loader.load()

# for document in documents:
#     print(f"Page Content: {document.page_content}")
#     print(f"Metadata: {document.metadata}")

client = AzureOpenAI(
    api_version= version,
    azure_endpoint= endpoint,
    api_key= key
)


## prompt

extract_Invoice_prompt = """
Extract appropriate fields and generate json as per the following schema and keep the json formatting as code:
    {
	"headers": {
		"vendorname": string
	},
	"orderdetails": {
		"documentDate": datetime, 
        // This refers to the invoice or document date. Use the field labeled "Invoice Date", "Date of Invoice", "Document Date", etc. 
        // Do not use "Ship Date", "Due Date", or unrelated dates. Format: YYYY-MM-DD.
		
        "vendorSoNumber": string,  
        // This is the Sales Order Number — not a shipment or delivery number. Look for fields labeled "Sales Order", "SO#", "Order#" etc. 
        // Ignore fields labeled "Shipment Number", "Shipping ID", etc.
		
        "vendorPoNumber": string,
        // This is the Purchase Order Number — usually labeled "Purchase Order", "PO#", etc. 
        // Do not extract shipment-related numbers.
		
        "endCustomerName": string,
	},
	"licensedetails": [
		{
			"partno": string,
            // This refers to the Part Number or SKU ID. Extract only values containing uppercase A–Z, numbers 0–9, or hyphens (-). 
            // Ignore non-English characters, Cyrillic text, or special characters.
			
            "quantity": int,
			
            "licenseKey": string,
            // This may not be explicitly labeled. If a string resembling a license key (usually alphanumeric, often with dashes or unique formatting) 
            // appears near the quantity or part number, extract it as licenseKey. If no such key is present nearby, return null.
			
            "startdate": datetime,
			"enddate": datetime
		}
	],
	"others": {
		"slcdetails": [
			{
				"slcId": string,
                // Just extract the slcid which is in alphanumeric format with hyphens.

				"carePlusExpDate": datetime,
				"carePremExpDate": datetime,
				"deviceQty": int
			}
		],
        // Optional. Only extract SLC details if they are present. 

		"SERIALS SHIPPED": [string]
        // Optional. If present, extract a list of serial numbers (strings). These may appear under headings like "Serials Shipped", "Serial Numbers" or similar
        // Extract the full string value associated with each serial, including any descriptive or parenthetical text
        // If not present, return an empty list.
	}
    }
"""


invoice_text = "\n".join(doc.page_content for doc in documents)
new_prompt = extract_Invoice_prompt +"\n\nInvoice Text:\n"+ invoice_text
    
#gpt-4.1
response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are a helpful assistant that extracts information from PDF files and returns output as valid JSON."},
        {"role": "user", "content": new_prompt}
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model= model_deployment
)

raw_text = response.choices[0].message.content

print(raw_text)


---------------------------------------------------------------------------
ClientAuthenticationError                 Traceback (most recent call last)
File <command-2315930052461248>, line 16
     11 file_path = "/Volumes/dev_datascience/volumes/ds-idp-volume/LM_PDFs/email 4.pdf"
     13 loader = AzureAIDocumentIntelligenceLoader(
     14     api_endpoint=endpoint, api_key=key, file_path=file_path, api_model="prebuilt-layout"
     15 )
---> 16 documents = loader.load()
     18 # for document in documents:
     19 #     print(f"Page Content: {document.page_content}")
     20 #     print(f"Metadata: {document.metadata}")
     22 client = AzureOpenAI(
     23     api_version= version,
     24     azure_endpoint= endpoint,
     25     api_key= key
     26 )

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-cae5c231-5dbc-456c-be18-1f30f633b5cd/lib/python3.11/site-packages/langchain_core/document_loaders/base.py:32, in BaseLoader.load(self)
     30 def load(self) -> list[Document]:
     31  